# 09 — fastText from scratch : subwords et généralisation OOV
**Navigation** : [<< Précédent — 08. Vecteurs de mots (word2vec)](08-Word2Vec-CBOW-SkipGram.ipynb) | [Index de la série](README.md) | **09 (ce notebook)** | [Suivant — 10. Naive Bayes >>](10-Classement-Texte-NaiveBayes-FromScratch.ipynb)

Ce carnet construit une variante pédagogique de fastText : le vecteur d'entrée est la somme des n-grammes de caractères connus. Contrairement au moteur de production, cette version n'utilise ni hachage ni vecteur atomique supplémentaire. Les mots absents du vocabulaire peuvent ainsi partager des paramètres avec les mots appris ; leur qualité reste à mesurer.

Le carnet 08 partage le corpus mais diffère aussi par la fenêtre, la dimension, les paires, l'objectif et le pas : il ne s'agit pas d'une ablation appariée.

**Version jumelle** : [Python](09-FastText-Subword-Vectors.ipynb).

**Objectifs** : construire des sous-mots, vérifier un gradient et mesurer la couverture hors vocabulaire.  
**Prérequis** : C#, vecteurs et descente de gradient. **Durée** : 75 minutes.

Le générateur aléatoire C# diffère de NumPy : les voisinages et les valeurs mesurées peuvent différer.


## 1. Le corpus et la décomposition en subwords

Le corpus synthétique est le même que celui du carnet 08 : 98 tokens et 36 mots. Les expériences diffèrent toutefois par plusieurs paramètres. Chaque mot est décomposé en n-grammes uniques de longueur 3 à 6 avec bornes `<` et `>`.

In [1]:
#r "nuget: MathNet.Numerics, 5.0.0"
using System;
using System.Linq;
using System.Collections.Generic;
using System.Text;
using System.Globalization;
using MathNet.Numerics.Distributions;
using Microsoft.DotNet.Interactive.Formatting;
record PlotSvg(string Content);
Formatter.Register<PlotSvg>((v,w)=>w.Write(v.Content),"text/html");
string F(double x)=>x.ToString("0.###",CultureInfo.InvariantCulture);
var rng=new Random(42); var normal=new Normal(0,0.1,rng);
string corpus="le roi porte la couronne sur le trone\nla reine porte la couronne sur le trone\nle roi regne dans le royaume\nla reine regne dans le royaume\nle cheval court dans le champ\nla jument court dans le champ\nle cheval danse sous un arbre\nla jument danse sous un arbre\nle chat miaule pres du chien\nle chaton miaule pres du chiot\nle chien aboie dans le jardin\nle chiot aboie dans le jardin\nla pomme est un fruit rouge\nla poire est un fruit vert\nun enfant mange la pomme\nun enfant mange la poire\n";
var tokens=corpus.ToLowerInvariant().Split((char[])null,StringSplitOptions.RemoveEmptyEntries);
var vocab=tokens.Distinct().OrderBy(x=>x,StringComparer.Ordinal).ToArray();var word2id=vocab.Select((w,i)=>(w,i)).ToDictionary(x=>x.w,x=>x.i);int V=vocab.Length,D=24;
Console.WriteLine($".NET {Environment.Version} | MathNet {typeof(Normal).Assembly.GetName().Version}");
Console.WriteLine($"tokens : {tokens.Length} | vocabulaire : {V} | {string.Join(", ",vocab)}");

Installed Packages MathNet.Numerics, 5.0.0

.NET 10.0.14 | MathNet 5.0.0.0


tokens : 98 | vocabulaire : 36 | aboie, arbre, champ, chat, chaton, cheval, chien, chiot, couronne, court, dans, danse, du, enfant, est, fruit, jardin, jument, la, le, mange, miaule, poire, pomme, porte, pres, regne, reine, roi, rouge, royaume, sous, sur, trone, un, vert


### Lecture du résultat

Le corpus synthétique compte **98 tokens pour 36 types**, avec des fréquences variables : certains mots n'apparaissent qu'une fois. Le vocabulaire trié rassemble animaux, plantes et verbes ; ce petit jeu permet d'inspecter les calculs et les voisinages sans prétendre mesurer une qualité générale d'embeddings.

In [2]:
string[] Subwords(string word,int minN=3,int maxN=6) {
    if(minN<1 || maxN<minN) throw new ArgumentException("Bornes invalides");
    string bounded="<"+word+">"; var seen=new HashSet<string>();
    for(int n=minN;n<=maxN;n++) for(int i=0;i+n<=bounded.Length;i++) seen.Add(bounded.Substring(i,n));
    return seen.OrderBy(x=>x,StringComparer.Ordinal).ToArray();
}
foreach(string w in new[]{"chaton","pomme","couronne"}) Console.WriteLine($"{w,-10} → {string.Join(", ",Subwords(w))}");
var subVocab=vocab.SelectMany(w=>Subwords(w)).Distinct().OrderBy(x=>x,StringComparer.Ordinal).ToArray();var sub2id=subVocab.Select((w,i)=>(w,i)).ToDictionary(x=>x.w,x=>x.i);int SV=subVocab.Length;
Console.WriteLine($"Mots : {V} | sous-mots : {SV} | ratio de lignes : {(double)SV/V:F1}");

chaton     → <ch, <cha, <chat, <chato, ato, aton, aton>, cha, chat, chato, chaton, hat, hato, haton, haton>, on>, ton, ton>


pomme      → <po, <pom, <pomm, <pomme, me>, mme, mme>, omm, omme, omme>, pom, pomm, pomme, pomme>


couronne   → <co, <cou, <cour, <couro, cou, cour, couro, couron, ne>, nne, nne>, onn, onne, onne>, our, ouro, ouron, ouronn, ron, ronn, ronne, ronne>, uro, uron, uronn, uronne


Mots : 36 | sous-mots : 418 | ratio de lignes : 11.6


### Lecture du résultat

`chaton` possède 18 n-grammes uniques de longueur 3 à 6 : les préfixes `<ch`, `<cha`, `<chat`, les suffixes `aton>`, `haton>` et des fragments internes. Le mot non borné `chaton` figure parmi eux, mais `<chaton>` est trop long pour cette plage. Deux mots partageant des fragments partagent des paramètres, sans garantie que leurs vecteurs entiers soient proches.

La table comporte **418 vecteurs de sous-mots** contre 36 mots : le rapport 11,6 compare des nombres de lignes, pas toute la mémoire des chaînes et dictionnaires.

## 2. Le modèle : un mot = somme de ses subwords

La table `S` fournit une ligne par sous-mot connu et le vecteur d'entrée est leur somme. La sortie calcule un softmax sur les mots cibles. Les carnets 08 et 09 partagent ce principe de prédiction, mais ici la fenêtre est symétrique, la dimension est 24 et les paires et le pas diffèrent ; les résultats ne constituent pas une ablation où seule l'entrée changerait.

In [3]:
var S=new double[SV,D];var Wp=new double[V,D];var bp=new double[V];
for(int i=0;i<SV;i++) for(int j=0;j<D;j++) S[i,j]=normal.Sample();
for(int i=0;i<V;i++) for(int j=0;j<D;j++) Wp[i,j]=normal.Sample();
double[] Embed(string word) {var result=new double[D];foreach(string sub in Subwords(word)) if(sub2id.TryGetValue(sub,out int row)) for(int j=0;j<D;j++) result[j]+=S[row,j];return result;}
Console.WriteLine($"Tables : S {SV}x{D}, Wp {V}x{D}, biais {V} ; mot sans fragment connu → vecteur nul");

Tables : S 418x24, Wp 36x24, biais 36 ; mot sans fragment connu → vecteur nul


La table de sous-mots et la couche de sortie sont initialisées. Vérifions maintenant la perte et ses gradients sur des cas bornés.

In [4]:
record SubGrad(double Loss,int[] Rows,double[] Dh,double[,] Wp,double[] Bp);
SubGrad FastGrad(string center,int target,double[,] table,double[,] output,double[] bias) {
    var rows=Subwords(center).Where(sub2id.ContainsKey).Select(s=>sub2id[s]).ToArray();var h=new double[D];foreach(int r in rows) for(int j=0;j<D;j++) h[j]+=table[r,j];
    var z=new double[V];for(int i=0;i<V;i++){z[i]=bias[i];for(int j=0;j<D;j++) z[i]+=output[i,j]*h[j];}
    double max=z.Max(), logZ=max+Math.Log(z.Sum(x=>Math.Exp(x-max)));var ds=z.Select(x=>Math.Exp(x-logZ)).ToArray();double loss=logZ-z[target];ds[target]-=1;
    var dwp=new double[V,D];var dh=new double[D];for(int i=0;i<V;i++)for(int j=0;j<D;j++){dwp[i,j]=ds[i]*h[j];dh[j]+=output[i,j]*ds[i];}
    return new SubGrad(loss,rows,dh,dwp,ds);
}
double NumCheck(string center,int target) {
    var g=FastGrad(center,target,S,Wp,bp);double maxError=0,eps=1e-5;
    void Check(ref double x,double expected){double old=x;x=old+eps;double p=FastGrad(center,target,S,Wp,bp).Loss;x=old-eps;double m=FastGrad(center,target,S,Wp,bp).Loss;x=old;maxError=Math.Max(maxError,Math.Abs(expected-(p-m)/(2*eps)));}
    foreach(int row in g.Rows.Take(4))for(int d=0;d<3;d++)Check(ref S[row,d],g.Dh[d]);
    foreach(int row in new[]{target,(target+1)%V}) {for(int d=0;d<3;d++)Check(ref Wp[row,d],g.Wp[row,d]);Check(ref bp[row],g.Bp[row]);}
    return maxError;
}
int targetEx=word2id["fruit"];double error=NumCheck("pomme",targetEx);if(error>1e-4)throw new Exception("Gradient incorrect");
Console.WriteLine($"Différences finies bornées : erreur maximale {error:E3}");
var sEx=new double[SV,D];var wEx=new double[V,D];var bEx=new double[V];int rowEx=sub2id[Subwords("pomme")[0]],otherEx=(targetEx+1)%V;sEx[rowEx,0]=1;wEx[targetEx,1]=1;bEx[targetEx]=-30;
var gEx=FastGrad("pomme",targetEx,sEx,wEx,bEx);double eps=1e-5,maxExtreme=0;
void CheckEx(ref double x,double expected){double old=x;x=old+eps;double p=FastGrad("pomme",targetEx,sEx,wEx,bEx).Loss;x=old-eps;double m=FastGrad("pomme",targetEx,sEx,wEx,bEx).Loss;x=old;maxExtreme=Math.Max(maxExtreme,Math.Abs(expected-(p-m)/(2*eps)));}
CheckEx(ref sEx[rowEx,1],gEx.Dh[1]);CheckEx(ref wEx[targetEx,0],gEx.Wp[targetEx,0]);CheckEx(ref wEx[otherEx,0],gEx.Wp[otherEx,0]);CheckEx(ref bEx[targetEx],gEx.Bp[targetEx]);CheckEx(ref bEx[otherEx],gEx.Bp[otherEx]);
if(!double.IsFinite(gEx.Loss)||maxExtreme>1e-7) throw new Exception("Fixture extrême incorrecte");
Console.WriteLine($"Logits extrêmes : erreur max S/Wp/biais {maxExtreme:E3}");

Différences finies bornées : erreur maximale 2.159E-011


Logits extrêmes : erreur max S/Wp/biais 3.786E-011


### Lecture du résultat

La sortie compare quelques composantes de `S` et `Wp` aux différences finies, ainsi que des composantes de biais et une ligne non cible. La fixture à logits extrêmes vérifie séparément `S`, `Wp` cible/non-cible et les biais. Ces tests bornés contrôlent les cas choisis ; ils ne prouvent pas l'absence d'erreur pour toute entrée.

### Exercice 1 — Vérifier le gradient de sortie

Vérifiez par différences finies les composantes de `Wp[target]`. Le paramètre `provider` permet de contrôler aussi un gradient volontairement perturbé, sans accepter une réponse constante.

In [5]:
// Exercice 1 : vérifier la ligne de sortie du gradient fourni.
bool? VerifierSortie(string center,int target,Func<string,int,double[,],double[,],double[],SubGrad> provider) {
    // TODO : différences finies sur Wp[target] et comparaison au provider.
    return null;
}
SubGrad Perturbe(string center,int target,double[,] s,double[,] w,double[] b) {var g=FastGrad(center,target,s,w,b);g.Wp[target,0]+=0.1;return g;}
var ex1=VerifierSortie("pomme",targetEx,FastGrad);
if(ex1==null) Console.WriteLine("Exercice 1 à compléter");
else Console.WriteLine(ex1==true && VerifierSortie("pomme",targetEx,Perturbe)==false ? "Fixtures gradient validées" : "Réponse incorrecte");

Exercice 1 à compléter


## 3. Entraînement + courbe de perte

Le skip-gram utilise une fenêtre **symétrique** de deux mots à gauche et à droite, qui traverse les retours à la ligne : 386 paires. Chaque époque mélange ces paires. Les lignes de `S` sont mises à jour pour les sous-mots du centre ; `Wp` reste dense.

In [6]:
var pairs=new List<(string Center,int Target)>();
for(int i=0;i<tokens.Length;i++) for(int j=Math.Max(0,i-2);j<Math.Min(tokens.Length,i+3);j++) if(i!=j) pairs.Add((tokens[i],word2id[tokens[j]]));
int[] Permutation(int n) {var a=Enumerable.Range(0,n).ToArray();for(int i=n-1;i>0;i--){int j=rng.Next(i+1);(a[i],a[j])=(a[j],a[i]);}return a;}
var losses=new List<double>();
for(int epoch=0;epoch<60;epoch++) {double total=0;foreach(int i in Permutation(pairs.Count)) {var pair=pairs[i];var g=FastGrad(pair.Center,pair.Target,S,Wp,bp);foreach(int row in g.Rows)for(int d=0;d<D;d++)S[row,d]-=0.1*g.Dh[d];for(int w=0;w<V;w++){bp[w]-=0.1*g.Bp[w];for(int d=0;d<D;d++)Wp[w,d]-=0.1*g.Wp[w,d];}total+=g.Loss;}losses.Add(total/pairs.Count);}
double reference=Math.Log(V);Console.WriteLine($"Paires {pairs.Count} | perte {losses[0]:F6} → {losses[^1]:F6} | ln(V)={reference:F6} | finale {(losses[^1]<reference?"sous":"au-dessus de")} la référence");
var svg=new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' width='760' height='380' viewBox='0 0 760 380'><rect width='760' height='380' fill='white'/><text x='380' y='24' text-anchor='middle' font-size='16'>Sous-mots — perte skip-gram</text>");
double ymax=Math.Ceiling(Math.Max(losses.Max(),reference)*2)/2,left=70,top=50,width=640,height=250;
for(int j=0;j<=5;j++){double y=top+height-j*height/5;svg.Append($"<line x1='{F(left)}' y1='{F(y)}' x2='{F(left+width)}' y2='{F(y)}' stroke='#ddd'/><text x='62' y='{F(y+4)}' text-anchor='end' font-size='12'>{F(ymax*j/5)}</text>");}
for(int j=0;j<=6;j++){double x=left+j*width/6;svg.Append($"<text x='{F(x)}' y='320' text-anchor='middle' font-size='12'>{j*10}</text>");}
svg.Append("<path d='M70 50 V300 H710' fill='none' stroke='black'/><text x='70' y='42' font-size='12'>perte moyenne (cross-entropie)</text><text x='390' y='344' text-anchor='middle' font-size='12'>époque</text>");
var points=string.Join(" ",losses.Select((v,i)=>$"{F(left+(i+1)*width/60)},{F(top+height-v/ymax*height)}"));double refY=top+height-reference/ymax*height;
svg.Append($"<polyline points='{points}' fill='none' stroke='#2465a8' stroke-width='2'/><line x1='70' y1='{F(refY)}' x2='710' y2='{F(refY)}' stroke='#555' stroke-dasharray='6 4'/><line x1='390' y1='358' x2='410' y2='358' stroke='#2465a8' stroke-width='2'/><text x='416' y='362' font-size='12'>perte</text><line x1='480' y1='358' x2='500' y2='358' stroke='#555' stroke-dasharray='6 4'/><text x='506' y='362' font-size='12'>ln(V) uniforme</text></svg>");display(new PlotSvg(svg.ToString()));

Paires 386 | perte 3.361688 → 2.929347 | ln(V)=3.583519 | finale sous la référence


Sous-mots — perte skip-gram 0 0.8 1.6 2.4 3.2 4 0 10 20 30 40 50 60 perte moyenne (cross-entropie) époque <polyline points='80.667,89.895 91.333,111.064 102,113.958 112.667,116.652 123.333,116.401 134,115.554 144.667,115.612 155.333,117.032 166,118.428 176.667,114.649 187.333,115.745 198,117.086 208.667,117.496 219.333,118.215 230,115.111 240.667,118.702 251.333,116.208 262,117.103 272.667,116.495 283.333,117.327 294,117.449 304.667,116.241 315.333,118.733 326,115.753 336.667,116.926 347.333,115.603 358,115.826 368.667,115.868 379.333,117.768 390,115.492 400.667,118.361 411.333,117.19 422,115.769 432.667,116.763 443.333,117.546 454,117.637 464.667,116.815 475.333,114.756 486,116.049 496.667,116.411 507.333,116.966 518,115.585 528.667,116.101 539.333,119.152 550,116.712 560.667,116.971 571.333,115.154 582,116.089 592.667,117.546 603.333,115.15 614,117.9 624.667,116.472 635.333,115.505 646,114.958 656.667,116.252 667.333,117.374 678,114.541 688.667,115.994 699.333,115.962 710,116.916' fill='none' stroke='#2465a8' stroke-width='2'/> perte ln(V) uniforme

### Lecture du résultat

La perte moyenne passe de **3,362 à 2,929** sur 60 époques, sous la référence uniforme `ln(36) = 3,584`. Cette référence est une cross-entropie, pas une accuracy. La courbe décrit une seule initialisation sur 386 paires ; elle ne mesure pas à elle seule une généralisation à d'autres corpus.

## 4. Sous-mots et mots OOV

Les quatre mots ci-dessous sont absents du vocabulaire d'entraînement. Le modèle peut sommer les sous-mots connus d'un mot nouveau ; la couverture et les voisins observés ne garantissent pas une proximité sémantique. Si aucun sous-mot n'est connu, ce notebook retourne un vecteur nul et ne fournit aucun voisin informatif.

In [7]:
double Cosine(double[] a,double[] b){double dot=0,na=0,nb=0;for(int i=0;i<D;i++){dot+=a[i]*b[i];na+=a[i]*a[i];nb+=b[i]*b[i];}return dot/(Math.Sqrt(na)*Math.Sqrt(nb)+1e-9);}
(string Word,double Score)[] NeighborsAny(string word,int k=5) {if(k<0)throw new ArgumentOutOfRangeException(nameof(k));var v=Embed(word);if(v.All(x=>x==0))return Array.Empty<(string,double)>();return vocab.Select(w=>(Word:w,Score:Cosine(v,Embed(w)))).OrderByDescending(x=>x.Score).Take(k).ToArray();}
string[] novel={"pommier","chatonne","couronnement","jardiner"};
foreach(string word in novel){var sw=Subwords(word);int known=sw.Count(sub2id.ContainsKey);Console.WriteLine($"{word,-14} → {string.Join(", ",NeighborsAny(word,4).Select(x=>$"{x.Word} ({x.Score:F2})"))} | couverture {known}/{sw.Length}");}
if(Subwords("").Length!=0 || Embed("zzzzzz").Any(x=>x!=0) || NeighborsAny("zzzzzz").Length!=0) throw new Exception("Contrat sous-mots absents incorrect");
Console.WriteLine("Mot sans fragment connu : vecteur nul, aucun voisin informatif");

pommier        → pomme (0.83), fruit (0.65), couronne (0.60), est (0.55) | couverture 6/22


chatonne       → chaton (0.87), du (0.74), chat (0.64), pres (0.57) | couverture 20/26


couronnement   → couronne (0.91), poire (0.78), fruit (0.67), reine (0.66) | couverture 28/42


jardiner       → jardin (0.94), poire (0.52), sur (0.43), couronne (0.41) | couverture 15/26


Mot sans fragment connu : vecteur nul, aucun voisin informatif


### Lecture du résultat

Les premiers voisins mesurés sont `pommier → pomme (0,83)`, `chatonne → chaton (0,87)`, `couronnement → couronne (0,91)` et `jardiner → jardin (0,94)`. Les couvertures valent respectivement **6/22**, **20/26**, **28/42** et **15/26** : `pommier` partage donc moins d'un tiers de ses fragments, et non « la plupart ». Ces dérivés ont été choisis pour leur proximité morphologique avec le corpus ; les scores décrivent ce run et ne prouvent pas une généralisation sémantique indépendante.

### Exercice 2 — Couverture OOV

Pour chaque mot nouveau, compter un succès si au moins une référence apparaît parmi ses `k` voisins. Mesurez la proportion ; une liste vide et `k = 0` renvoient zéro ; un `k` négatif est refusé par `ValueError` (Python) ou `ArgumentOutOfRangeException` (C#). Les fixtures distinguent plusieurs couvertures.

In [8]:
// Exercice 2 : proportion de références présentes parmi les k voisins.
double? Couverture(string[] mots,Dictionary<string,HashSet<string>> refs,Func<string,int,string[]> voisins,int k=3) {
    // TODO : proportion de hits, entrée vide ou k=0 → 0.
    return null;
}
var fixtureVoisins=new Dictionary<string,string[]>{{"a",new[]{"x","y"}},{"b",new[]{"z","x"}},{"c",new[]{"y","z"}}};
string[] VoisinsFixture(string w,int k)=>fixtureVoisins[w].Take(k).ToArray();
var refsEx=new Dictionary<string,HashSet<string>>{{"a",new(){"x"}},{"b",new(){"x"}},{"c",new(){"x"}}};
var ex2=Couverture(new[]{"a","b","c"},refsEx,VoisinsFixture,1);
if(ex2==null) Console.WriteLine("Exercice 2 à compléter");
else {var autre=Couverture(new[]{"a","b","c"},refsEx,VoisinsFixture,2);var vide=Couverture(Array.Empty<string>(),new(),VoisinsFixture,1);var zero=Couverture(new[]{"a"},refsEx,VoisinsFixture,0);
 bool valid=autre.HasValue && vide.HasValue && zero.HasValue && double.IsFinite(ex2.Value) && double.IsFinite(autre.Value) && double.IsFinite(vide.Value) && double.IsFinite(zero.Value) && Math.Abs(ex2.Value-1.0/3)<1e-9 && Math.Abs(autre.Value-2.0/3)<1e-9 && vide.Value==0 && zero.Value==0;
 try { Couverture(new[]{"a"},refsEx,VoisinsFixture,-1); valid=false; }
 catch(ArgumentOutOfRangeException) { }
 Console.WriteLine(valid?"Fixtures couverture validées":"Réponse incorrecte");}

Exercice 2 à compléter


## 5. Confrontation structurelle avec 08

Les mots nouveaux sont absents de la table de mots du carnet 08. Ici on peut construire une somme de sous-mots connus. Les quatre dérivés partagent des fragments avec des mots du corpus ; la présence du radical en top 3 est une mesure morphologique descriptive, pas un test de généralisation sémantique ni une comparaison expérimentale appariée.

In [9]:
var derives=new Dictionary<string,string>{{"pommier","pomme"},{"chatonne","chaton"},{"couronnement","couronne"},{"jardiner","jardin"}};int hits=0;
foreach(var (word,radical) in derives){var top=NeighborsAny(word,3).Select(x=>x.Word).ToArray();bool hit=top.Contains(radical);if(hit)hits++;Console.WriteLine($"{word,-14} vocabulaire mot : {(word2id.ContainsKey(word)?"oui":"non")} | top3 {string.Join(", ",top)} | radical {radical} : {(hit?"présent":"absent")}");}
var seenFleur=Subwords("fleur").Where(sub2id.ContainsKey).ToArray();Console.WriteLine($"{hits}/{derives.Count} radicaux en top3 sur ce banc");
Console.WriteLine($"Contrôle fleur : {seenFleur.Length}/{Subwords("fleur").Length} fragment(s) connus [{string.Join(", ",seenFleur)}] ; top3 {string.Join(", ",NeighborsAny("fleur",3).Select(x=>x.Word))}");

pommier        vocabulaire mot : non | top3 pomme, fruit, couronne | radical pomme : présent


chatonne       vocabulaire mot : non | top3 chaton, du, chat | radical chaton : présent


couronnement   vocabulaire mot : non | top3 couronne, poire, fruit | radical couronne : présent


jardiner       vocabulaire mot : non | top3 jardin, poire, sur | radical jardin : présent


4/4 radicaux en top3 sur ce banc


Contrôle fleur : 1/14 fragment(s) connus [ur>] ; top3 sur, porte, reine


### Lecture du résultat

Les quatre dérivés (`pommier`, `chatonne`, `couronnement`, `jardiner`) sont hors vocabulaire et leur radical est dans le top 3 sur cette exécution. Ce résultat concerne quatre exemples choisis, avec des réglages différents du carnet 08 ; il ne constitue pas une comparaison appariée.

Le contrôle `fleur` partage **un fragment sur 14**, `ur>`, et son top 3 est `sur`, `porte`, `reine`. Il ne s'agit donc pas d'un vecteur nul ni d'un contrôle sans fragment ; cette faible couverture n'établit pas de généralisation sémantique. Un mot sans aucun fragment connu produit en revanche le vecteur nul et aucun voisin informatif.

## Exercices — récapitulatif

Les exercices répartis portent sur le [gradient de sortie](#exercice-1--vérifier-le-gradient-de-sortie), la [couverture OOV](#exercice-2--couverture-oov) et le [coût des sous-mots](#exercice-3--coût-du-vocabulaire-de-sous-mots). Les fonctions sont à compléter et leurs fixtures distinguent plusieurs cas, dont les entrées vides.

### Exercice 3 — Coût du vocabulaire de sous-mots

Dédupliquez les mots et leurs n-grammes, puis rapportez le nombre de lignes de chaque table et leur ratio. Pour une entrée vide renvoyez `(0, 0, 0)` ; ce ratio ne compte pas la mémoire des dictionnaires.

In [10]:
// Exercice 3 : nombre de sous-mots, mots distincts et ratio.
(int Subs,int Mots,double Ratio)? CoutSousMots(string[] mots,int minN=3,int maxN=6) {
    // TODO : dédupliquer et renvoyer (0,0,0) pour une entrée vide.
    return null;
}
var ex3=CoutSousMots(new[]{"a","ab","a"});
if(ex3==null) Console.WriteLine("Exercice 3 à compléter");
else {var autre=CoutSousMots(new[]{"ab","ab"},3,3);var vide=CoutSousMots(Array.Empty<string>());
 bool valid=autre.HasValue && vide.HasValue && ex3.Value.Subs==4 && ex3.Value.Mots==2 && double.IsFinite(ex3.Value.Ratio) && Math.Abs(ex3.Value.Ratio-2)<1e-9 && autre.Value.Subs==2 && autre.Value.Mots==1 && Math.Abs(autre.Value.Ratio-2)<1e-9 && vide.Value.Subs==0 && vide.Value.Mots==0 && vide.Value.Ratio==0;
 Console.WriteLine(valid?"Fixtures coût validées":"Réponse incorrecte");}

Exercice 3 à compléter


## À retenir

- Cette variante pédagogique représente l'entrée par la somme de n-grammes connus de longueur 3 à 6 ; elle ne reprend ni le hachage ni un vecteur atomique supplémentaire du moteur de production.
- Le corpus contient 98 tokens et 36 mots ; la table comporte **418 vecteurs de sous-mots** à 24 dimensions. Le ratio 418/36 compte des lignes et ne mesure pas toute la mémoire.
- Les 386 paires utilisent une fenêtre symétrique ±2, traversant les lignes. La perte softmax est calculée par logsumexp ; les différences finies vérifient un ensemble borné de composantes, dont des logits extrêmes.
- Sur ce run, les quatre dérivés placent leur radical en top 3, avec des couvertures 6/22, 20/26, 28/42 et 15/26. `fleur` partage `ur>` (1/14), tandis qu'un mot sans fragment connu donne un vecteur nul et aucun voisin informatif.
- Les réglages du carnet 08 diffèrent aussi par la fenêtre, les paires, la dimension, l'objectif et le pas : les deux expériences ne constituent pas une ablation appariée.

## Vérifiez votre compréhension

1. Pourquoi un mot hors vocabulaire peut-il recevoir un vecteur ?
2. Que se passe-t-il si aucun de ses sous-mots n'est connu ?
3. Que mesure le contrôle `fleur` ?
4. Peut-on attribuer les écarts avec le carnet 08 à la seule représentation d'entrée ?

<details><summary>Réponses</summary>

1. Ses sous-mots connus partagent des lignes de la table apprises sur d'autres mots.
2. Leur somme est nulle ; les cosinus sont nuls et aucun voisin informatif n'est renvoyé.
3. `fleur` possède un fragment connu sur 14, `ur>` : c'est une faible couverture, pas une absence de vecteur.
4. Non. Les fenêtres, dimensions, paires, objectifs et pas diffèrent aussi ; il faudrait un protocole apparié.

</details>

## Pour aller plus loin

**Carnet suivant** : [10-Classement-Texte-NaiveBayes-FromScratch.ipynb](10-Classement-Texte-NaiveBayes-FromScratch.ipynb) — les représentations deviennent des features : classer un document par ses mots, from scratch à nouveau.

- **Lectures** : Bojanowski, Grave, Joulin & Mikolov 2017, « Enriching Word Vectors with Subword Information » (l'article fastText) ; Mikolov et al. 2013, « Efficient Estimation of Word Representations in Vector Space » (skip-gram et CBOW).
- **Dans le dépôt** : le carnet 08 (word2vec from scratch), qui partage le corpus mais pas tous les réglages expérimentaux ; la série `RAG-et-Memoire-Semantique` pour le devenir de ces vecteurs dans la recherche sémantique.
